In [1]:
!pip install -U langgraph -q
!pip install -U langgraph-checkpoint-sqlite -q
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 625.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 8.4 MB/s eta 0:00:00


In [2]:
!pip install langchain-openai

In [3]:
import uuid
import sqlite3
from langgraph.store.sqlite import SqliteStore
import os
from google.colab import userdata
# Defina sua chave de API da OpenAI aqui
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
conn = sqlite3.connect("memoria_compras.db", check_same_thread=False)
store = SqliteStore(conn, index={"embed": "openai:text-embedding-3-small", "dims": 1536})
store.setup()


In [7]:
try:
    conn.rollback()
except Exception:
    pass




In [8]:
# ========================= CARLOS =========================
# 1. Definindo o namespace que será utilizado para salvar os dados
namespace = ("memories", "carlos")

# 2. Recuperando registros existentes ou buscando por similaridade semântica usando search
resultados_busca = store.search(namespace, query="Quais produtos o Carlos quer comprar?", limit=1)
print("=== Busca inicial do Carlos ===")
for item in resultados_busca:
    print(f"ID: {item.key} | Conteúdo: {item.value}")

# Insere um dado passando (namespace, chave_única, dicionário_com_os_dados)
store.put(("memories", "carlos"), "user_preference", {"data": "Carlos quer um celular com bateria duradoura e câmera de qualidade para fotos."})

# 1. Em vez de usar uma chave fixa como "user_preference", geramos chaves únicas (UUID)
chave_1 = str(uuid.uuid4())
chave_2 = str(uuid.uuid4())

# 2. Salvando múltiplas informações no mesmo namespace
store.put(("memories", "carlos"), chave_1, {"data": "Carlos procura um tablet para estudar, ler PDFs e fazer anotações."})
store.put(("memories", "carlos"), chave_2, {"data": "Carlos deseja uma TV 4K de 55 polegadas para assistir a filmes e séries."})

# 3. Agora, ao buscar, você verá os registros deste namespace!
resultados = store.search(("memories", "carlos"), query="Eletrônicos e equipamentos para comprar", limit=1)
print(f"\n=== Total de registros encontrados para Carlos: {len(resultados)} ===")
for item in resultados:
    print(f"Chave: {item.key} -> {item.value}")

=== Busca inicial do Carlos ===
ID: user_preference | Conteúdo: {'data': 'Carlos quer um celular com bateria duradoura e câmera de qualidade para fotos.'}

=== Total de registros encontrados para Carlos: 1 ===
Chave: user_preference -> {'data': 'Carlos quer um celular com bateria duradoura e câmera de qualidade para fotos.'}


In [9]:
# ========================= MARIANA =========================
# 1. Definindo o namespace da segunda pessoa no MESMO banco SQLite
namespace = ("memories", "mariana")

# 2. Buscando registros já existentes
resultados_busca = store.search(namespace, query="Quais produtos a Mariana quer comprar?", limit=1)
print("\n=== Busca inicial da Mariana ===")
for item in resultados_busca:
    print(f"ID: {item.key} | Conteúdo: {item.value}")

# Insere uma preferência com a mesma chave fixa, mas em outro namespace
store.put(("memories", "mariana"), "user_preference", {"data": "Mariana prefere um smartphone leve, compacto e com boa câmera para viagens."})

# 1. Gerando chaves únicas (UUID) para as outras memórias da Mariana
chave_3 = str(uuid.uuid4())
chave_4 = str(uuid.uuid4())

# 2. Salvando outras informações no namespace da Mariana
store.put(("memories", "mariana"), chave_3, {"data": "Mariana quer um tablet com caneta digital para desenhar e criar ilustrações."})
store.put(("memories", "mariana"), chave_4, {"data": "Mariana busca uma TV com tela grande e boa qualidade de imagem para assistir a esportes."})

# 3. Buscando os registros salvos para Mariana
resultados = store.search(("memories", "mariana"), query="Eletrônicos e equipamentos para comprar", limit=1)
print(f"\n=== Total de registros encontrados para Mariana: {len(resultados)} ===")
for item in resultados:
    print(f"Chave: {item.key} -> {item.value}")


=== Busca inicial da Mariana ===
ID: 54957507-4636-4881-8830-fa3bf9846069 | Conteúdo: {'data': 'Mariana quer um tablet com caneta digital para desenhar e criar ilustrações.'}

=== Total de registros encontrados para Mariana: 1 ===
Chave: 6304a8a5-f2e8-4420-82da-131987595b07 -> {'data': 'Mariana busca uma TV com tela grande e boa qualidade de imagem para assistir a esportes.'}
